# Full KIS embedding ablation — OpenCLIP vs SigLIP2

Standalone Kaggle notebook chạy toàn bộ 58 KIS `Status=OK` trong `aic2026_all_confirmed.csv`. Notebook chỉ dùng hai collection image embedding có sẵn trên Zilliz, không tải hoặc benchmark Qwen, không dùng OpenAI/Groq/Elasticsearch.

Kaggle Secrets bắt buộc: `MILVUS_URI`, `MILVUS_TOKEN`. Dataset bắt buộc: `dev_search_local.db` tại `/kaggle/input/datasets/thnhlcdng/ablation/`.

In [ ]:
# ===== CONFIG + DEPENDENCIES =====
import os, re, gc, json, math, time, sqlite3, shutil, subprocess, sys
from pathlib import Path

SOURCE_DB = Path('/kaggle/input/datasets/thnhlcdng/ablation/dev_search_local.db')
DB_PATH = Path('/kaggle/working/dev_search_local.db')
REPO_URL = 'https://github.com/zintomvn/Multimodal-Retrieval.git'
BRANCH = 'experiment/embedding-ablation-l01-l25'
REPO = Path('/kaggle/working/embedding-ablation-source')
OUTPUT = Path('/kaggle/working/embedding_ablation_openclip_siglip2_all_results')
TOP_K, SEARCH_LIMIT, SEARCH_LEVEL = 100, 1000, 10
PERSPECTIVE_COUNTS = [3, 5, 7]

if not SOURCE_DB.is_file():
    raise FileNotFoundError(f'Missing database: {SOURCE_DB}')
if not DB_PATH.is_file() or DB_PATH.stat().st_size != SOURCE_DB.stat().st_size:
    shutil.copy2(SOURCE_DB, DB_PATH)
with sqlite3.connect(DB_PATH) as db_check:
    if db_check.execute('PRAGMA quick_check').fetchone()[0] != 'ok':
        raise RuntimeError(f'Invalid SQLite database: {DB_PATH}')
if REPO.exists():
    subprocess.run(['git','fetch','origin',BRANCH], cwd=REPO, check=True)
    subprocess.run(['git','checkout','-B',BRANCH,'FETCH_HEAD'], cwd=REPO, check=True)
else:
    subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','pymilvus>=2.5,<2.7','open_clip_torch>=2.32','transformers>=4.57.3','sentencepiece','accelerate'], check=True)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
MILVUS_URI = secrets.get_secret('MILVUS_URI')
MILVUS_TOKEN = secrets.get_secret('MILVUS_TOKEN')
if not MILVUS_URI or not MILVUS_TOKEN:
    raise RuntimeError('Missing Kaggle Secrets MILVUS_URI/MILVUS_TOKEN')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('DB:', DB_PATH)
print('Output:', OUTPUT)


In [ ]:
# ===== LOAD ALL CONFIRMED KIS + CREATE 7 FROZEN ENGLISH VIEWS =====
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

GT_PATH = REPO / 'data/experiments/aic_2026_groundtruth/aic2026_all_confirmed.csv'
gt = pd.read_csv(GT_PATH, encoding='utf-8-sig')
def pick(row, *names):
    for name in names:
        if name in row.index and pd.notna(row[name]): return str(row[name]).strip()
    return ''
def parse_frames(raw):
    return tuple(int(x) for x in re.findall(r'\d+', str(raw)))
queries = []
for _, row in gt.iterrows():
    qid = pick(row, 'Query ID', 'Original Query ID')
    text = pick(row, 'Query Text', 'Query')
    video = pick(row, 'GT Video ID')
    frames = parse_frames(pick(row, 'GT Frame ID(s)', 'GT Frame ID'))
    status = pick(row, 'Status')
    task = pick(row, 'Task Type')
    if not re.search(r'(?:^|::)query-.*-kis$', qid, re.I): continue
    if status.upper() != 'OK' or task.upper() != 'KIS': continue
    if not text or not re.match(r'^L\d+_', video, re.I) or not frames: continue
    queries.append({'query_id':qid,'text':text,'video_id':video,'frame_ids':frames})
if len(queries) != 58:
    raise RuntimeError(f'Expected 58 confirmed KIS queries, got {len(queries)}')
with sqlite3.connect(DB_PATH) as con:
    frame_lookup = {str(k):(str(v),int(f)) for k,v,f in con.execute('SELECT keyframe_id, video_id, frame_idx FROM keyframes WHERE is_media_present=1')}
if not frame_lookup:
    raise RuntimeError('No keyframes found in SQLite')
print('Queries:', len(queries), '| Full keyframes:', len(frame_lookup))

def source_views(text, n=7):
    text = re.sub(r'\s+', ' ', text).strip()
    parts = [p.strip(' ,.;:-') for p in re.split(r'[.!?;:]|\b(?:sau đó|tiếp theo|cuối cùng|đồng thời|trong khi|bên cạnh|phía trước|phía sau)\b', text, flags=re.I) if p.strip(' ,.;:-')]
    words = text.split(); candidates = list(parts)
    for windows in (n, n-1, n+1):
        for i in range(windows):
            start, end = i*len(words)//windows, (i+1)*len(words)//windows
            if end > start: candidates.append(' '.join(words[start:end]))
    out=[]; seen=set()
    for item in candidates:
        key=item.casefold()
        if item and key not in seen: out.append(item); seen.add(key)
        if len(out)==n: return out
    raise ValueError(f'Query too short for seven views: {text}')
source_by_query = {q['query_id']:source_views(q['text']) for q in queries}
full_source = {q['query_id']:q['text'] for q in queries}
all_source = list(dict.fromkeys([*full_source.values(), *(x for views in source_by_query.values() for x in views)]))
translator_id = 'facebook/nllb-200-distilled-600M'
tok = AutoTokenizer.from_pretrained(translator_id, src_lang='vie_Latn')
translator = AutoModelForSeq2SeqLM.from_pretrained(translator_id, torch_dtype=torch.float16).to('cuda').eval()
translations = {}
for start in range(0, len(all_source), 16):
    batch = all_source[start:start+16]
    inputs = tok(batch, return_tensors='pt', padding=True, truncation=True, max_length=256).to('cuda')
    with torch.inference_mode():
        generated = translator.generate(**inputs, forced_bos_token_id=tok.convert_tokens_to_ids('eng_Latn'), max_new_tokens=128)
    for src,dst in zip(batch,tok.batch_decode(generated,skip_special_tokens=True)):
        translations[src] = re.sub(r'\s+', ' ', dst).strip()
full_queries = {qid:translations[text] for qid,text in full_source.items()}
perspectives = {qid:[translations[x] for x in views] for qid,views in source_by_query.items()}
(OUTPUT/'full_queries.json').write_text(json.dumps(full_queries,ensure_ascii=False,indent=2),encoding='utf-8')
(OUTPUT/'perspectives.json').write_text(json.dumps(perspectives,ensure_ascii=False,indent=2),encoding='utf-8')
del translator,tok; gc.collect(); torch.cuda.empty_cache()
print('Frozen perspectives:', OUTPUT/'perspectives.json')


In [ ]:
# ===== VALIDATE COLLECTIONS; DIRECT TEXT EMBEDDING + FULL-CORPUS SEARCH =====
import numpy as np
from pymilvus import MilvusClient

SPECS = {
 'openclip': {'collection':'keyframe_embeddings_clip_vith14_quickgelu_dfn5b_v2','dim':1024,'model':'ViT-H-14-quickgelu','pretrained':'dfn5b'},
 'siglip2': {'collection':'keyframe_embeddings_siglip2_so400m16_384_webli_openclip_1152_v1','dim':1152,'model':'ViT-SO400M-16-SigLIP2-384','pretrained':'webli'},
}
client = MilvusClient(uri=MILVUS_URI, token=MILVUS_TOKEN, timeout=120)
collection_rows = {}
for name,spec in SPECS.items():
    if not client.has_collection(collection_name=spec['collection']): raise RuntimeError(f'Missing collection: {spec["collection"]}')
    desc=client.describe_collection(collection_name=spec['collection'])
    vf=next(f for f in desc.get('fields',[]) if f.get('name')=='vector')
    dim=int((vf.get('params') or {}).get('dim') or vf.get('dim') or 0)
    rows=int(client.get_collection_stats(collection_name=spec['collection']).get('row_count') or 0)
    if dim!=spec['dim']: raise RuntimeError(f'{name}: expected dim {spec["dim"]}, got {dim}')
    if rows!=len(frame_lookup): raise RuntimeError(f'{name}: collection rows {rows} != SQLite keyframes {len(frame_lookup)}')
    collection_rows[name]=rows
    print(name, spec['collection'], 'dim=',dim,'rows=',rows)
if len(set(collection_rows.values())) != 1: raise RuntimeError(f'Corpus mismatch: {collection_rows}')
for name,spec in SPECS.items():
    sample=client.query(collection_name=spec['collection'],filter='id != ""',limit=1,output_fields=['id','vector'])
    if len(sample)!=1: raise RuntimeError(f'{name}: cannot read an audit vector')
    sample_id=str(sample[0]['id'])
    vector=np.asarray(sample[0]['vector'],dtype='float32')
    hits=client.search(collection_name=spec['collection'],anns_field='vector',data=[vector.tolist()],limit=1,output_fields=['id','keyframe_id'],search_params={'metric_type':'COSINE','params':{'level':SEARCH_LEVEL}},consistency_level='Strong',timeout=120)[0]
    entity=hits[0].get('entity') or {}; hit_id=str(hits[0].get('id') or entity.get('keyframe_id') or '')
    score=float(hits[0].get('distance',hits[0].get('score',0)))
    if hit_id!=sample_id or score<0.999: raise RuntimeError(f'{name}: ANN self-check failed: expected={sample_id}, got={hit_id}, score={score}')
    print(name,'ANN self-check=PASS','score=',score)

def resolve_hit(hit):
    entity=hit.get('entity') or {}
    for key in ('canonical_keyframe_id','mapped_keyframe_id','keyframe_id','frame_id'):
        candidate=str(entity.get(key) or '')
        if candidate in frame_lookup: return candidate
    candidate=str(hit.get('id') or '')
    return candidate if candidate in frame_lookup else None
def gt_rank(ranked_ids,query):
    for rank,kid in enumerate(ranked_ids,1):
        video,frame=frame_lookup[kid]
        if video==query['video_id'] and frame in query['frame_ids']: return rank
    return -1
def search_views(collection,vectors):
    data=np.asarray(vectors,dtype='float32').tolist()
    fields=['frame_id','video_id','keyframe_id','canonical_keyframe_id','mapped_keyframe_id','frame_idx']
    raw=client.search(collection_name=collection,anns_field='vector',data=data,limit=TOP_K,output_fields=fields,search_params={'metric_type':'COSINE','params':{'level':SEARCH_LEVEL}},consistency_level='Strong',timeout=120)
    scores={}
    for hits in raw:
        for hit in hits:
            kid=resolve_hit(hit)
            if kid: scores[kid]=max(scores.get(kid,-1e9),float(hit.get('distance',hit.get('score',0))))
    return [kid for kid,_ in sorted(scores.items(),key=lambda x:x[1],reverse=True)[:TOP_K]]

records=[]
for model_name,spec in SPECS.items():
    print('\n===',model_name,'===')
    import open_clip
    model,_,_=open_clip.create_model_and_transforms(spec['model'],pretrained=spec['pretrained'],device='cuda')
    tokenizer=open_clip.get_tokenizer(spec['model']); model.eval()
    def encode(texts):
        tokens=tokenizer(texts).to('cuda')
        with torch.inference_mode(), torch.autocast('cuda',dtype=torch.float16): vec=model.encode_text(tokens)
        return torch.nn.functional.normalize(vec.float(),dim=-1).cpu().numpy()
    probe=encode(['a test image query'])
    if probe.shape!=(1,spec['dim']) or not np.isfinite(probe).all(): raise RuntimeError(f'{model_name} bad text vectors: {probe.shape}')
    configurations=[('full_query',1)]+[('perspective',n) for n in PERSPECTIVE_COUNTS]
    for qi,q in enumerate(queries,1):
        for mode,n in configurations:
            texts=[full_queries[q['query_id']]] if mode=='full_query' else perspectives[q['query_id']][:n]
            started=time.perf_counter(); vectors=encode(texts); ranked=search_views(spec['collection'],vectors); latency=(time.perf_counter()-started)*1000
            records.append({'model':model_name,'query_mode':mode,'n':n,'query_id':q['query_id'],'gt_video_id':q['video_id'],'gt_frame_ids':';'.join(map(str,q['frame_ids'])),'rank':gt_rank(ranked,q),'latency_ms':round(latency,3),'returned':len(ranked)})
        print(f'{model_name}: {qi}/{len(queries)}')
    del model,tokenizer; gc.collect(); torch.cuda.empty_cache()
per_query=pd.DataFrame(records)
per_query.to_csv(OUTPUT/'per_query.csv',index=False,encoding='utf-8-sig')
print('Search complete:',len(per_query),'measurements')


In [ ]:
# ===== METRICS + PAPER-STYLE RANK TABLE + KAGGLE OUTPUT ZIP =====
def summarize(group):
    ranks=group['rank'].astype(int); found=ranks[ranks>0]; lat=group['latency_ms'].astype(float)
    row={'queries':len(group),'MRR':float((1.0/found).sum()/len(group)) if len(group) else 0.0,'mean_latency_ms':float(lat.mean()),'p50_latency_ms':float(lat.quantile(.5)),'p95_latency_ms':float(lat.quantile(.95)),'misses_at_100':int((ranks<0).sum())}
    for k in (1,5,10,50,100): row[f'Recall@{k}']=float(((ranks>0)&(ranks<=k)).mean())
    return pd.Series(row)
summary=per_query.groupby(['model','query_mode','n'],sort=False).apply(summarize,include_groups=False).reset_index()
summary.to_csv(OUTPUT/'summary.csv',index=False,encoding='utf-8-sig')
aliases={q['query_id']:f'q{i}' for i,q in enumerate(queries,1)}
rank_table=per_query.pivot_table(index=['model','query_mode','n'],columns='query_id',values='rank',aggfunc='first').reset_index()
rank_table=rank_table.rename(columns=aliases)
ordered=['model','query_mode','n']+[aliases[q['query_id']] for q in queries]
rank_table=rank_table[ordered]
rank_table.to_csv(OUTPUT/'rank_table.csv',index=False,encoding='utf-8-sig')
lines=['| '+' | '.join(rank_table.columns)+' |','|'+'|'.join(['---']*len(rank_table.columns))+'|']
for _,row in rank_table.iterrows(): lines.append('| '+' | '.join(str(x) for x in row.tolist())+' |')
lines += ['', 'Query mapping:']+[f'- {alias}: `{qid}`' for qid,alias in aliases.items()]
(OUTPUT/'rank_table.md').write_text('\n'.join(lines)+'\n',encoding='utf-8')
config={'scope':'all confirmed KIS','query_count':len(queries),'models':SPECS,'top_k':TOP_K,'search_limit':SEARCH_LIMIT,'search_level':SEARCH_LEVEL,'consistency_level':'Strong','perspective_counts':PERSPECTIVE_COUNTS,'similarity':'COSINE','frame_tolerance':0,'perspective_generator':'facebook/nllb-200-distilled-600M deterministic contiguous chunks'}
(OUTPUT/'run_config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
archive=shutil.make_archive('/kaggle/working/embedding_ablation_openclip_siglip2_all_results','zip',OUTPUT)
display(summary)
print('DONE — output:',OUTPUT)
print('ZIP:',archive)
